# Star Schema Warehouse

Builds dimensional model from gold layer for BI tools.

**Source:** workspace.gold_chocolate  
**Target:** workspace.warehouse_chocolate

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, row_number, dense_rank
)
from pyspark.sql.window import Window
import os


GOLD_CATALOG = "workspace"
GOLD_SCHEMA = "gold_chocolate"
WAREHOUSE_CATALOG = "workspace"
WAREHOUSE_SCHEMA = "warehouse_chocolate"





✓ Imports loaded
✓ Source: workspace.gold_chocolate
✓ Target: workspace.warehouse_chocolate
✓ Export path: /Workspace/Users/alihesham345@gmail.com/chocolate_warehouse_export


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}")

✓ Schema workspace.warehouse_chocolate ready


In [0]:


df_gold_fact_sales = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_fact_sales")
df_gold_dim_date = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_date")
df_gold_dim_product = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_product")
df_gold_dim_store = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_store")
df_gold_dim_geography = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_geography")




📊 Loading Gold tables...
  • Fact Sales: 1,000,000 rows
  • Dim Date: 731 rows
  • Dim Product: 202 rows
  • Dim Store: 100 rows
  • Dim Geography: 6 rows


In [0]:



df_dim_date = df_gold_dim_date.select(
    col("date_id").alias("date_key"),
    col("sale_date").alias("date"),
    col("year"),
    col("quarter"),
    col("month"),
    col("month_name"),
    col("day"),
    col("day_name"),
    col("day_of_week"),
    col("week_of_year"),
    col("is_weekend")
).orderBy("date_key")

dim_date_path = f"{WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}.dim_date"
df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_date_path)



df_dim_product = df_gold_dim_product.select(
    col("product_id").alias("product_key"),
    col("product_name"),
    col("product_name_clean"),
    col("brand"),
    col("brand_clean"),
    col("category").alias("product_category"),
    col("category_clean"),
    col("total_quantity_sold"),
    col("total_revenue"),
    col("total_profit"),
    col("avg_profit_margin_pct"),
    col("total_transactions")
).orderBy("product_key")

dim_product_path = f"{WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}.dim_product"
df_dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_product_path)



df_dim_store = df_gold_dim_store.select(
    col("store_id").alias("store_key"),
    col("store_name"),
    col("store_name_clean"),
    col("city"),
    col("city_clean"),
    col("country"),
    col("country_clean"),
    col("store_type"),
    col("store_type_clean"),
    col("total_quantity_sold"),
    col("total_revenue"),
    col("total_profit"),
    col("avg_profit_margin_pct"),
    col("total_sales_count"),
    col("unique_customers")
).orderBy("store_key")

dim_store_path = f"{WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}.dim_store"
df_dim_store.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_store_path)



df_dim_geography = df_gold_dim_geography.select(
    col("country_id").alias("geography_key"),
    col("country").alias("country_code"),
    col("country_name"),
    col("total_quantity_sold"),
    col("total_revenue"),
    col("total_profit"),
    col("avg_profit_margin_pct"),
    col("total_transactions"),
    col("store_count"),
    col("customer_count")
).orderBy("geography_key")

dim_geography_path = f"{WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}.dim_geography"
df_dim_geography.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_geography_path)


print(f"Created {df_dim_date.count():,} date, {df_dim_product.count():,} product, {df_dim_store.count():,} store, {df_dim_geography.count():,} geography dimensions")


⭐ Building Star Schema Dimensions...
  📅 Creating DIM_DATE...
    ✓ workspace.warehouse_chocolate.dim_date: 731 rows
  🍫 Creating DIM_PRODUCT...
    ✓ workspace.warehouse_chocolate.dim_product: 202 rows
  🏪 Creating DIM_STORE...
    ✓ workspace.warehouse_chocolate.dim_store: 100 rows
  🌍 Creating DIM_GEOGRAPHY...
    ✓ workspace.warehouse_chocolate.dim_geography: 6 rows

✓ All dimension tables created!


In [0]:



df_fact_sales = df_gold_fact_sales.alias("f") \
    .join(
        df_dim_date.select("date_key", col("date").alias("join_date")),
        col("f.sale_date") == col("join_date"),
        "left"
    ) \
    .join(
        df_dim_product.select("product_key", col("product_name").alias("join_product")),
        col("f.product_name") == col("join_product"),
        "left"
    ) \
    .join(
        df_dim_store.select("store_key", col("store_name").alias("join_store")),
        col("f.store_name") == col("join_store"),
        "left"
    ) \
    .join(
        df_dim_geography.select("geography_key", col("country_code").alias("join_country")),
        col("f.country") == col("join_country"),
        "left"
    )


df_fact_final = df_fact_sales.select(
    col("f.sale_id").alias("sale_key"),
    

    col("date_key"),
    col("product_key"),
    col("store_key"),
    col("geography_key"),
    

    col("f.order_id"),
    col("f.sale_date"),
    col("f.product_name"),
    col("f.brand"),
    col("f.category"),
    col("f.store_name"),
    col("f.city"),
    col("f.country"),
    col("f.store_type"),
    

    col("f.customer_id"),
    col("f.age"),
    col("f.gender"),
    col("f.loyalty_member"),
    

    col("f.quantity"),
    col("f.unit_price"),
    col("f.discount"),
    col("f.discount_amount"),
    col("f.revenue"),
    col("f.cost"),
    col("f.profit"),
    col("f.profit_margin_pct"),
    

    col("f.is_valid")
).orderBy("sale_key")




fact_sales_path = f"{WAREHOUSE_CATALOG}.{WAREHOUSE_SCHEMA}.fact_sales"
df_fact_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fact_sales_path)

print(f"Fact table: {df_fact_final.count():,} rows -> {fact_sales_path}")


🎯 Creating FACT_SALES table...
  • Fact table rows: 8,786,033
  • Columns: 27
  ✓ Saved to: workspace.warehouse_chocolate.fact_sales


In [0]:



fact_count = df_fact_final.count()
date_nulls = df_fact_final.filter(col("date_key").isNull()).count()
product_nulls = df_fact_final.filter(col("product_key").isNull()).count()
store_nulls = df_fact_final.filter(col("store_key").isNull()).count()
geography_nulls = df_fact_final.filter(col("geography_key").isNull()).count()

print(f"\nValidation: {fact_count:,} records, {date_nulls} null dates, {product_nulls} null products, {store_nulls} null stores, {geography_nulls} null geographies")




✅ Validating Star Schema Relationships...

  • Total fact records: 8,786,033
  • Null date_key: 0
  • Null product_key: 9764
  • Null store_key: 0
  • Null geography_key: 0

⚠️ Warning: Some null foreign keys detected. Check dimension joins.


In [0]:



volume_path = "/Volumes/workspace/warehouse_chocolate/export"
csv_file_path = f"{volume_path}/fact_sales.csv"




df_fact_final.coalesce(1) \
    .write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(csv_file_path)

print(f"Exported {df_fact_final.count():,} rows to {csv_file_path}")


📤 Exporting Fact Table to CSV...
  • Writing to: /Volumes/workspace/warehouse_chocolate/export/fact_sales.csv
  • Records to export: 8,786,033
  ✓ CSV export complete!
  ✓ File location: /Volumes/workspace/warehouse_chocolate/export/fact_sales.csv

  💡 To download: Open Catalog Explorer → workspace → warehouse_chocolate → export volume


In [0]:
print("\nStar schema warehouse summary:")
print(f"Fact: {spark.table(fact_sales_path).count():,} rows")
print(f"Dimensions: {spark.table(dim_date_path).count():,} dates, {spark.table(dim_product_path).count():,} products, {spark.table(dim_store_path).count():,} stores, {spark.table(dim_geography_path).count():,} geographies")
print(f"CSV export: {csv_file_path}")


STAR SCHEMA DATA WAREHOUSE - COMPLETE

🎯 FACT TABLE:
  • workspace.warehouse_chocolate.fact_sales: 8,786,033 rows

⭐ DIMENSION TABLES:
  • workspace.warehouse_chocolate.dim_date: 731 rows
  • workspace.warehouse_chocolate.dim_product: 202 rows
  • workspace.warehouse_chocolate.dim_store: 100 rows
  • workspace.warehouse_chocolate.dim_geography: 6 rows

📤 EXPORT:
  • CSV export: /Volumes/workspace/warehouse_chocolate/export/fact_sales.csv

✅ Star Schema Warehouse is ready for BI consumption!


In [0]:
%sql
-- Revenue and profit by category, store, and country

SELECT 
    dp.product_category,
    ds.store_name,
    ds.city,
    dg.country_name,
    COUNT(fs.sale_key) as total_transactions,
    SUM(fs.quantity) as total_quantity,
    ROUND(SUM(fs.revenue), 2) as total_revenue,
    ROUND(SUM(fs.profit), 2) as total_profit,
    ROUND(AVG(fs.profit_margin_pct), 2) as avg_profit_margin_pct
FROM workspace.warehouse_chocolate.fact_sales fs
INNER JOIN workspace.warehouse_chocolate.dim_product dp ON fs.product_key = dp.product_key
INNER JOIN workspace.warehouse_chocolate.dim_store ds ON fs.store_key = ds.store_key
INNER JOIN workspace.warehouse_chocolate.dim_geography dg ON fs.geography_key = dg.geography_key
GROUP BY dp.product_category, ds.store_name, ds.city, dg.country_name
ORDER BY total_revenue DESC
LIMIT 20

product_category,store_name,city,country_name,total_transactions,total_quantity,total_revenue,total_profit,avg_profit_margin_pct
Praline,Chocolate Store 74,Sydney,USA,25167,75942,642651.72,257655.37,40.09
Praline,Chocolate Store 50,New York,CANADA,25086,75719,642133.18,256969.98,40.0
Praline,Chocolate Store 80,Sydney,AUSTRALIA,25220,75616,641547.05,256539.79,40.02
Praline,Chocolate Store 61,London,FRANCE,24750,74197,641487.15,256766.74,39.99
Praline,Chocolate Store 98,New York,AUSTRALIA,24883,75400,639234.23,256000.29,40.02
Praline,Chocolate Store 85,Melbourne,USA,24645,74577,639108.38,255530.16,40.03
Praline,Chocolate Store 12,London,FRANCE,24838,74582,638303.16,255544.58,40.04
Praline,Chocolate Store 39,New York,UK,24795,74721,638119.85,254458.98,39.9
Praline,Chocolate Store 93,Sydney,UK,24780,74725,638107.22,255119.94,40.02
Praline,Chocolate Store 27,Sydney,GERMANY,24935,74665,638022.17,255926.93,40.12
